In [ ]:
""" Two distinct error types evaluated separately, and kept clearly labeled throughout:
Table S2 - per-expert false positives (flagged, no condition) vs. true negatives,
    age/BMI profile on the condition-positive + pure-control population (matches
    Figure S3's confusion matrices), plus other conditions present.
Table S3 - gate false positives/negatives vs. true negatives/positives
    --> this is the model's actual trained task."""
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

from help.ckm_utils import CONDITIONS, COND_NAMES

In [ ]:
UNDERPOWERED_N = 20   # minimum n (flagged vs not-flagged, or FP/FN vs TN/TP) to
                       # attempt a Mann-Whitney/age-BMI comparison at all; below this the
                       # comparison is skipped, not reported with an unstable p-value

In [ ]:
POP_PATH = "analytic_population.pkl"
MOE_PATH = "Results_MoE/moe_participant_results.pkl"
OUT_DIR = "Results_MoE_Misclassifications"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
def load_data():
    df = pd.read_pickle(POP_PATH)
    moe = pd.read_pickle(MOE_PATH)
    for d in (df, moe):
        d["gender"] = d["gender"].str.lower().str.strip()
    assert df["participant_id"].is_unique and moe["participant_id"].is_unique

    clinical = df[["participant_id", "age", "bmi"]].copy()

    flags = df.set_index("participant_id")[list(CONDITIONS.values())].eq(1)
    flags.columns = COND_NAMES

    return df, moe, clinical, flags

def _fdr_family(out, cols, suffix_in="_p", suffix_out="_q"):
    """Pool the given p-value columns into a single BH-FDR correction and
    write back the corresponding q-value columns on `out`"""
    pooled_p = out[cols].stack()
    if not len(pooled_p):
        return
    _, q_vals, _, _ = multipletests(pooled_p.to_numpy(), method="fdr_bh")
    q_series = pd.Series(q_vals, index=pooled_p.index)
    for col in cols:
        out[col.replace(suffix_in, suffix_out)] = q_series.xs(col, level=1).reindex(out.index)

df, moe, clinical, flags = load_data()

In [ ]:
def per_expert_flagging_profile(moe, flags, clinical):
    """FP vs TN per expert, Figure S3's confusion matrices"""
    base = moe.merge(flags.reset_index(), on="participant_id", how="left")
    base = base.merge(clinical[["participant_id", "age", "bmi"]], on="participant_id", how="left")

    rows = []
    for gender, gdf in base.groupby("gender"):
        for expert in COND_NAMES:
            pool = gdf.loc[gdf[expert] | (gdf["burden_score"] == 0)]
            no_target = pool.loc[~pool[expert]]          
            flagged_mask = no_target[f"flag_rate_{expert}"] >= 0.5
            flagged, not_flagged = no_target.loc[flagged_mask], no_target.loc[~flagged_mask]
            if len(flagged) < UNDERPOWERED_N or len(not_flagged) < UNDERPOWERED_N:
                continue

            row = {"gender": gender, "expert": expert,
                   "n_flagged": len(flagged), "n_not_flagged": len(not_flagged)}

            for var in ["age", "bmi"]:
                x, y = flagged[var].dropna(), not_flagged[var].dropna()
                if len(x) >= UNDERPOWERED_N and len(y) >= UNDERPOWERED_N:
                    _, p = mannwhitneyu(x, y, alternative="two-sided")
                    row[f"{var}_mean_flagged"], row[f"{var}_sd_flagged"] = x.mean(), x.std()
                    row[f"{var}_mean_not_flagged"], row[f"{var}_sd_not_flagged"] = y.mean(), y.std()
                    row[f"{var}_p"] = p
            rows.append(row)

    out = pd.DataFrame(rows)
    demo_cols = [c for c in out.columns if c in ("age_p", "bmi_p")]
    _fdr_family(out, demo_cols)
    return out


expert_profile = per_expert_flagging_profile(moe, flags, clinical)
expert_profile.to_csv(os.path.join(OUT_DIR, "TableS2_expert_flagging_profile.csv"), index=False)

display_cols = [
    "gender", "expert", "n_flagged", "n_not_flagged",
    "age_mean_flagged", "age_sd_flagged", "age_mean_not_flagged", "age_sd_not_flagged", "age_q",
    "bmi_mean_flagged", "bmi_sd_flagged", "bmi_mean_not_flagged", "bmi_sd_not_flagged", "bmi_q",
]
print("Table S2 - per-expert flagging profile: age/BMI (FP vs TN)")
display(expert_profile[display_cols].round(3))

In [ ]:
def expert_confusion_matrices(moe, flags):
    """Sensitivity/specificity/precision per expert"""
    base = moe.merge(flags.reset_index(), on="participant_id", how="left")
    rows = []
    for gender, gdf in base.groupby("gender"):
        for expert in COND_NAMES:
            pool = gdf.loc[gdf[expert] | (gdf["burden_score"] == 0)]
            pred = pool[f"flag_rate_{expert}"] >= 0.5
            actual = pool[expert]
            tp, fp = int((pred & actual).sum()), int((pred & ~actual).sum())
            fn, tn = int((~pred & actual).sum()), int((~pred & ~actual).sum())
            rows.append({
                "gender": gender, "expert": expert,
                "TP": tp, "FP": fp, "FN": fn, "TN": tn, "n": len(pool),
                "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
                "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
                "precision": tp / (tp + fp) if (tp + fp) else np.nan,
            })
    return pd.DataFrame(rows)

expert_cm = expert_confusion_matrices(moe, flags)

expert_cm.to_csv(os.path.join(OUT_DIR, "TableS2_expert_confusion_matrices.csv"), index=False)
print("Table S2 - per-expert confusion matrices")
display(expert_cm)

In [ ]:
def composite_confusion(moe):
    rows = []
    for gender, gdf in moe.groupby("gender"):
        pred, actual = gdf["flag_rate_composite"] >= 0.5, gdf["y_any"].astype(bool)
        tp, fp = int((pred & actual).sum()), int((pred & ~actual).sum())
        fn, tn = int((~pred & actual).sum()), int((~pred & ~actual).sum())
        rows.append({"gender": gender, "TP": tp, "FP": fp, "FN": fn, "TN": tn, "n": len(gdf),
                     "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
                     "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
                     "precision": tp / (tp + fp) if (tp + fp) else np.nan})
    return pd.DataFrame(rows)


composite_cm = composite_confusion(moe)
composite_cm.to_csv(os.path.join(OUT_DIR, "TableS3_composite_confusion_matrix.csv"), index=False)
print("Table S3 - gate confusion matrix (y_any: has >=1 condition vs. pure control)")

display(composite_cm)

In [ ]:
def per_expert_comorbidity_profile(moe, flags):
    """Comorbidity enrichment: among the cohort's FP vs TN for each expert, 
    how much more/less common are the OTHER conditions."""
    base = moe.merge(flags.reset_index(), on="participant_id", how="left")
    rows = []
    for gender, gdf in base.groupby("gender"):
        for expert in COND_NAMES:
            no_target = gdf.loc[~gdf[expert]]
            flagged_mask = no_target[f"flag_rate_{expert}"] >= 0.5
            flagged, not_flagged = no_target.loc[flagged_mask], no_target.loc[~flagged_mask]
            if len(flagged) < UNDERPOWERED_N or len(not_flagged) < UNDERPOWERED_N:
                continue

            row = {"gender": gender, "expert": expert}
            for other in COND_NAMES:
                if other == expert:
                    continue
                pf = flagged[other].mean() * 100
                pn = not_flagged[other].mean() * 100
                a, b = pf / 100 * len(flagged), (1 - pf / 100) * len(flagged)
                c, d = pn / 100 * len(not_flagged), (1 - pn / 100) * len(not_flagged)
                zero_cell = (a == 0) or (b == 0) or (c == 0) or (d == 0)
                if zero_cell:
                    a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
                row[f"{other}_pct_flagged"] = pf
                row[f"{other}_pct_not_flagged"] = pn
                row[f"{other}_OR"] = (a / b) / (c / d)
                ct = pd.crosstab(no_target[f"flag_rate_{expert}"] >= 0.5, no_target[other])
                if ct.shape == (2, 2):
                    _, p, _, _ = chi2_contingency(ct)
                    row[f"{other}_p"] = p
            rows.append(row)

    out_df = pd.DataFrame(rows)
    p_cols = [c for c in out_df.columns if c.endswith("_p")]
    if p_cols:
        pooled_p = out_df[p_cols].stack()
        _, q_vals, _, _ = multipletests(pooled_p.to_numpy(), method="fdr_bh")
        q_series = pd.Series(q_vals, index=pooled_p.index)
        for col in p_cols:
            out_df[col.replace("_p", "_q")] = q_series.xs(col, level=1).reindex(out_df.index)
    return out_df


comorbidity_profile = per_expert_comorbidity_profile(moe, flags)
comorbidity_profile.to_csv(os.path.join(OUT_DIR, "TableS2_expert_comorbidity_profile.csv"), index=False)
print("Table S2 (comorbidity enrichment) - FP vs TN, other-condition odds ratios")
display(comorbidity_profile)